# Connect to Qdrant Cloud

Import the qdrant client and create a connection to your Qdrant Cloud cluster using your cluster URL and API key.

In [5]:
import os

from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, Document, PointStruct, VectorParams

load_dotenv()

collection_name = "items"

# Connect to Qdrant Cloud
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    cloud_inference=True,
    check_compatibility=False,
)

# Create collection

In [ ]:
# create collection
client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

# Populate the collection

In [7]:
from DATA02 import menu_items

model = "sentence-transformers/all-MiniLM-L6-v2"

# points generator
points = []
for i, menu_item in enumerate(menu_items):
    point = PointStruct(
        id=i,
        vector=Document(
            text=f"{menu_item[0]} {menu_item[1]}",
            model=model,
        ),
        payload={
            "item_name": menu_item[0],
            "description": menu_item[1],
            "price": menu_item[2],
            "category": menu_item[3],
        },
    )
    points.append(point)

# upsert points to collection
client.upsert(
    collection_name=collection_name,
    points=points,
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

# Search the Menu Items

In [ ]:
# Generate query embedding
query_text = "vegetarian dishes"

# Search for similar menu items
results = client.query_points(
    collection_name=collection_name,
    query=Document(
        text=query_text,
        model=model,
    ),
    with_payload=True,
    limit=5,
)

# Print results
for result in results.points:
    payload = result.payload

    if payload is None:
        continue

    item_name = payload.get("item_name", "N/A")
    description = payload.get("description", "N/A")
    price = payload.get("price", "N/A")

    print(f"Item: {item_name}")
    print(f"Score: {result.score}")
    print(f"Description: {description[:150]}...")
    print(f"Price: {price}")
    print("---")

Item: Greek Salad
Score: 0.5085151
Description: Romaine lettuce tomatoes cucumbers kalamata olives feta cheese red onion with lemon oregano dressing...
Price: $10.50
---
Item: Shrimp Tacos
Score: 0.50280845
Description: Three soft tacos with grilled shrimp cabbage slaw chipotle aioli and fresh lime...
Price: $13.00
---
Item: Oyster Po Boy
Score: 0.4853553
Description: Fried oysters with lettuce tomato pickles and remoulade on french bread...
Price: $15.50
---
Item: Stuffed Bell Peppers
Score: 0.482386
Description: Roasted bell peppers filled with rice vegetables herbs and melted cheese...
Price: $13.75
---
Item: Grilled Salmon Fillet
Score: 0.47120366
Description: Wild-caught Atlantic salmon grilled with lemon butter and fresh herbs served with seasonal vegetables...
Price: $24.50
---
